In [ ]:
from jax import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from numpyro import distributions as dist
from numpyro import infer
import arviz as az
import pickle

from estival.sampling import tools as esamp

from emu_renewal.inputs import DATA_PATH, MOB_MAP, VAR_MAP, get_multicountry_df_from_who_data, get_hosp_series_from_owid_data
from emu_renewal.inputs import get_seroprev, filter_seroprev, get_multivars_country_data, get_row_proportions
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import get_spaghetti, get_spagh_df_from_dict, get_output_dir
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
analysis = "mob"
country = "France"
message = "Simplify again, fixed CDR"

In [ ]:
# Analysis type
mobility = analysis == "mob"

# WHO indicators
cases_data = get_multicountry_df_from_who_data("New_cases", [country])
deaths_data = get_multicountry_df_from_who_data("New_deaths", [country])

# Hospitalisation data
hosp_data = get_hosp_series_from_owid_data("Daily hospital occupancy", country)

# Variant data
var_data = get_row_proportions(get_multivars_country_data(VAR_MAP, country))
var_data["eu"] = var_data["eu1"] + var_data["eu2"]
var_data = var_data[["eu", "alpha"]]

# Mobility
mob = pd.read_csv(DATA_PATH / f"mobility/{MOB_MAP[country]}_mob_data.csv", index_col=0)
mob.index = pd.to_datetime(mob.index)
non_resi_mob = mob.loc[:, mob.columns!="residential"]
model_mob = non_resi_mob.mean(axis=1).rolling(7).mean().dropna() if mobility else None

In [ ]:
# Specify fixed parameters and get calibration data
proc_update_freq = 10
init_time = 50
pop = 68e6
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
data_start = analysis_start + timedelta(14)
init_start = analysis_start - timedelta(init_time)
init_end = analysis_start - timedelta(1)
select_data = cases_data.loc[data_start: analysis_end, country]
select_deaths = deaths_data.loc[data_start: analysis_end, country]
select_vars = var_data.loc[data_start: analysis_end]
hosp_data = hosp_data[data_start: analysis_end: 7]
init_data = cases_data[country].resample("D").asfreq().interpolate().loc[init_start: init_end] / 7.0
seroprev = filter_seroprev(get_seroprev(), country, analysis_start, analysis_end)
seroprev.index -= timedelta(days=14)

In [ ]:
# Define model and fitter
alpha_seed_times = [analysis_start, analysis_start + timedelta(days=10)]
seed_times = [alpha_seed_times]
proc_fitter = CosineMultiCurve()
strains = ["eu", "alpha"]
model_args = [
    pop, 
    analysis_start, 
    analysis_end, 
    proc_update_freq, 
    proc_fitter, 
    GammaDens(), 
    init_time, 
    init_data, 
    GammaDens(), 
    GammaDens(), 
    strains, 
    "eu", 
    seed_times, 
    0.0, 
]
model = MultiStrainModel(*model_args + [model_mob])

In [ ]:
# Define parameter ranges
priors = {
    "gen_mean": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "gen_sd": dist.TruncatedNormal(2.5, 0.5, low=1.0),
    "cdr": 0.3, # dist.Beta(9.0, 21.0),
    "ifr": 0.02, #dist.Beta(10.0, 900.0),
    "rt_init": dist.Normal(0.0, 0.5),
    "report_mean": dist.TruncatedNormal(8.0, 0.5, low=1.0),
    "report_sd": dist.TruncatedNormal(3.0, 0.5, low=1.0),
    "death_mean": 21.0, #dist.TruncatedNormal(21.0, 5.0, low=1.0),
    "death_sd": 10.0, #dist.TruncatedNormal(10.0, 2.5, low=1.0),
    "admit_mean": 13.0, #dist.TruncatedNormal(13.0, 3.0, low=1.0),
    "admit_sd": 6.0, #dist.TruncatedNormal(6.0, 1.0, low=1.0),
    "stay_mean": 14.5, #dist.TruncatedNormal(14.5, 1.5, low=1.0),
    "stay_sd": 7.0, #dist.TruncatedNormal(7.0, 1.0, low=1.0),
    "har": 0.03, #dist.Beta(6, 200),
    "cross_immunity": 0.5, #dist.Beta(5, 5),
    "alpha_relinfect": 1.8, #dist.TruncatedNormal(1.25, 0.15, low=1.0),
    "shared_dispersion": dist.HalfNormal(0.5),
}

In [ ]:
# Define calibration and calib data
model_var_data = var_data.loc[(analysis_start < var_data.index) & (var_data.index < analysis_end), "eu"]
calib_data = {
    "weekly_cases": StandardDispTarget(select_data, weight=40.0),
    # "weekly_deaths": StandardDispTarget(select_deaths, weight=20.0),
    # "prop_eu": StandardDispTarget(model_var_data, weight=20.0),
    # "seropos": StandardDispTarget(seroprev, weight=20.0),
    # "occupancy": StandardDispTarget(hosp_data, weight=20.0),
}
calib = StandardCalib(model, priors, calib_data, proc_dispersion=dist.HalfNormal(0.5))

In [ ]:
# Run calibration
analysis_time = datetime.now().strftime("%d%m%Y_%H%M")
kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.5))
mcmc = infer.MCMC(kernel, num_chains=2, num_samples=1000, num_warmup=1000)
mcmc.run(random.PRNGKey(0))

In [ ]:
# Get and store outputs
out_dir = get_output_dir(country, analysis, analysis_time)
idata = az.from_dict(mcmc.get_samples(True))
idata.to_netcdf(out_dir / "idata.nc")
idata_sampled = az.extract(idata, num_samples=50)
sample_params = esamp.xarray_to_sampleiterator(idata_sampled)
spaghetti = get_spagh_df_from_dict(get_spaghetti(calib, sample_params))
spaghetti.to_hdf(out_dir / "spaghetti.h5", key="s")
updates = pd.DataFrame(sample_params.components["proc"], columns=model.epoch.index_to_dti(model.x_proc_vals)).T
updates.to_hdf(out_dir / "updates.h5", key="u")
pickle.dump(priors, open(get_output_dir(country, analysis, analysis_time) / "priors.pkl", "wb"))
for t, target in calib.targets.items():
    target.data.to_hdf(out_dir / f"target_{t}.h5", key=t)
with open(out_dir / "description.txt", "w") as file:
    file.write(message)

In [ ]:
analysis

In [ ]:
analysis_time

In [ ]:
az.summary(idata)

In [ ]:
model.strain_map

In [ ]:
az.plot_trace(idata, compact=False);